# CHG — Análisis Raster con Python
## Día 3 · Módulo A: NumPy — el lenguaje de los rasters

**Curso de Python para el Análisis Espacial**
Confederación Hidrográfica del Guadalquivir

---

> **Idea fuerza del día:** un raster (una imagen Landsat, un MDT, una máscara de agua…) es internamente **un array de NumPy**. Aprender NumPy es aprender a hacer teledetección.

## Contenidos

1. ¿Por qué NumPy en SIG/teledetección?
2. Creación de arrays
3. Atributos (ndim, shape, dtype) → leídos como propiedades de un raster
4. Indexación y *slicing* — recortar ventanas
5. Álgebra elemento a elemento — la base del NDVI
6. Reshape, stack y concatenación — apilar bandas
7. Funciones universales (`np.log`, `np.sqrt`, `np.where`…)
8. Estadística por eje (`axis`)
9. **Máscaras booleanas** — la base de NDWI binarizado, *no data*, clasificación
10. Mini-ejercicios contextualizados (NDVI, máscara de agua, clorofila)


## 1. ¿Por qué NumPy en SIG/teledetección?

Una imagen Landsat tiene, por ejemplo, **7711 × 7581 píxeles** y **6 bandas**. Son **~350 millones de números**. Si los recorremos con un `for` de Python, tardamos minutos. Con NumPy, segundos.

NumPy ofrece:
- **Arrays N-dimensionales** (`ndarray`): 2D = una banda; 3D = (bandas, alto, ancho).
- **Operaciones vectorizadas** — `(nir - red) / (nir + red)` se aplica píxel a píxel sin bucles.
- **Máscaras booleanas** — `agua = ndwi > 0.3` te devuelve un raster binario en una línea.

Casi todo el ecosistema geoespacial (rasterio, xarray, rioxarray, scikit-image, GDAL, GEE en Python…) trabaja **internamente con `ndarray`**.


In [ ]:
import numpy as np
print(f'NumPy {np.__version__}')

---
## 2. Creación de arrays

### 2.1 Desde listas (uso ocasional, típicamente con pocos valores)


In [ ]:
# Vector de valores NDVI medidos en 5 píxeles
ndvi_pixeles = np.array([0.12, 0.55, 0.78, 0.04, -0.10])
print(ndvi_pixeles)
print('Tipo:', ndvi_pixeles.dtype)


In [ ]:
# Un mini-raster 3x3 (escribiéndolo a mano)
mini_raster = np.array([
    [0.10, 0.12, 0.08],   # fila 0
    [0.45, 0.52, 0.50],   # fila 1
    [0.78, 0.82, 0.80],   # fila 2
])
print(mini_raster)
print('shape:', mini_raster.shape)   # (filas, columnas)


### 2.2 Arrays inicializados (lo más habitual)

| Función              | Para qué sirve en raster                              |
|----------------------|--------------------------------------------------------|
| `np.zeros(shape)`    | Crear una máscara o un raster vacío                    |
| `np.ones(shape)`     | Inicializar pesos, multiplicadores                     |
| `np.full(shape, v)`  | Rellenar con valor `nodata`                            |
| `np.arange(n)`       | Series temporales, índices                             |
| `np.linspace(a,b,n)` | Generar bordes para clasificar                         |
| `np.random.rand(*s)` | Datos sintéticos para probar algoritmos                |


In [ ]:
# Máscara vacía con la misma forma que el mini-raster
mascara = np.zeros(mini_raster.shape, dtype='uint8')
print(mascara)
print('dtype:', mascara.dtype)   # uint8 = 1 byte por píxel → ahorra MUCHA memoria


In [ ]:
# Raster relleno con valor "nodata" típico
nodata_val = -9999
relleno = np.full((4, 4), nodata_val, dtype='int16')
print(relleno)


In [ ]:
# Simulamos una banda Landsat de 5x5 píxeles con reflectancia 0-1
np.random.seed(42)
banda_simulada = np.random.rand(5, 5).round(2)
print(banda_simulada)


---
## 3. Atributos del array → propiedades del raster

Cuando abramos un raster con `rasterio` y pidamos `src.read()`, lo que obtenemos es un **`ndarray`**. Estos atributos serán nuestro pan de cada día:


In [ ]:
# Simulamos un raster multibanda Landsat: (6 bandas, 5 filas, 5 columnas)
np.random.seed(0)
landsat_simulado = np.random.rand(6, 5, 5).astype('float32')

print('ndim:   ', landsat_simulado.ndim,    '→ nº de dimensiones (3 = multibanda)')
print('shape:  ', landsat_simulado.shape,   '→ (bandas, alto, ancho)')
print('size:   ', landsat_simulado.size,    '→ nº total de píxeles × bandas')
print('dtype:  ', landsat_simulado.dtype,   '→ tipo de dato; float32 = 4 bytes/píxel')
print('nbytes: ', landsat_simulado.nbytes,  '→ memoria ocupada en bytes')


> ⚠️ **Importante**: el `dtype` determina el peso del raster. Una Landsat completa en `uint16` (2 bytes) ocupa la mitad que en `float32` (4 bytes). Solo convertimos a float cuando vamos a hacer divisiones (p. ej. al calcular NDVI), si no, se queda en su tipo entero.


In [ ]:
# Convertir tipo de dato (cast). Necesario antes de divisiones para evitar truncamiento entero
entero = np.array([[1, 2], [3, 4]], dtype='uint8')
flotante = entero.astype('float32')
print('Entero:   ', entero, entero.dtype)
print('Flotante: ', flotante, flotante.dtype)

# Si dividimos enteros, perdemos decimales
print('\nDivisión entera:  ', np.array([1], dtype='uint8') / np.array([3], dtype='uint8'))
# NumPy promueve a float64 automáticamente al dividir, pero si lo hubiéramos hecho con // sí truncaría


---
## 4. Indexación y *slicing*

Slicing en NumPy = recortar una ventana del raster sin abrirlo entero.


![axis](https://courses.spatialthoughts.com/images/python_foundation/pandas_axis.png)

En 2D: `array[fila, columna]`. En 3D: `array[banda, fila, columna]`.


In [ ]:
# Volvemos al mini-raster 3x3
mini_raster = np.array([
    [0.10, 0.12, 0.08],
    [0.45, 0.52, 0.50],
    [0.78, 0.82, 0.80],
])

print('Píxel (0,1):           ', mini_raster[0, 1])
print('Fila 0 entera:         ', mini_raster[0])
print('Columna 1 entera:      ', mini_raster[:, 1])
print('Subraster 2x2 (sup-der):\n', mini_raster[0:2, 1:3])


> 💡 **`np.arange(n)`** — el "rango" de NumPy: devuelve un array con los enteros `[0, 1, 2, ..., n-1]`. Es el equivalente al `range()` de Python pero devuelve un `ndarray` (en vez de un iterador), por lo que puedes hacerle `.reshape()`, slicing, álgebra, etc. directamente. Lo usamos mucho para generar "rasters de prueba" rellenos de valores predecibles.
>
> - `np.arange(12)` → `[0,1,2,3,4,5,6,7,8,9,10,11]`
> - `np.arange(0, 1, 0.1)` → `[0.0, 0.1, ..., 0.9]` (con paso decimal)


In [ ]:
# Recorte de una ventana 3x3 dentro de un raster grande
raster_grande = np.arange(100).reshape(10, 10)
print('Raster 10x10:')
print(raster_grande)

# Ventana centrada en (5,5), tamaño 3x3
ventana = raster_grande[4:7, 4:7]
print('\nVentana 3x3:')
print(ventana)


In [ ]:
# En un raster multibanda, seleccionar una banda concreta
# landsat_simulado.shape = (6, 5, 5)
banda_NIR = landsat_simulado[3]    # asumimos banda 4 (índice 3) = NIR
print('Forma de la banda NIR:', banda_NIR.shape)
print(banda_NIR)


---
## 5. Álgebra elemento a elemento — la base del NDVI

Las operaciones aritméticas se aplican **píxel a píxel**, sin escribir un solo bucle. Esto es lo que hace posible que un NDVI sobre una imagen entera sea **una sola línea de código**.


In [ ]:
# Simulamos 2 bandas pequeñas (rojo y NIR) de 4x4 píxeles
rojo = np.array([
    [0.05, 0.06, 0.08, 0.10],
    [0.07, 0.04, 0.05, 0.09],
    [0.15, 0.18, 0.20, 0.22],
    [0.30, 0.32, 0.31, 0.30],
], dtype='float32')

nir = np.array([
    [0.05, 0.04, 0.06, 0.08],   # agua (NIR bajo)
    [0.55, 0.52, 0.58, 0.50],   # vegetación sana
    [0.25, 0.28, 0.30, 0.27],   # suelo
    [0.40, 0.45, 0.42, 0.41],   # vegetación moderada
], dtype='float32')

ndvi = (nir - rojo) / (nir + rojo)
print(ndvi.round(2))


**Lee la matriz:** los valores cercanos a 0 son agua, ~0.8 vegetación sana, ~0.3 suelo desnudo o vegetación poco vigorosa.


In [ ]:
# Operadores básicos sobre arrays — todos vectorizados
a = np.array([1.0, 2.0, 3.0])
b = np.array([10.0, 20.0, 30.0])
print('a + b =', a + b)
print('b / a =', b / a)
print('a ** 2 =', a ** 2)
print('-a    =', -a)


---
## 6. Reshape, stack y concatenación — apilar bandas


In [ ]:
# reshape: cambiar la forma sin tocar los datos
v = np.arange(12)
print('Vector original (12,):', v)
print('Reshape a (3,4):\n', v.reshape(3, 4))
print('Reshape a (2,2,3) — como 2 bandas 2x3:\n', v.reshape(2, 2, 3))


In [ ]:
# np.stack — apilar bandas en un raster multibanda
banda1 = np.full((3, 3), 1, dtype='uint8')
banda2 = np.full((3, 3), 2, dtype='uint8')
banda3 = np.full((3, 3), 3, dtype='uint8')

# eje 0 → quedan en forma (bandas, alto, ancho) → como rasterio espera
stack = np.stack([banda1, banda2, banda3], axis=0)
print('shape:', stack.shape)
print(stack)


In [ ]:
# np.dstack — apila por el último eje → forma (alto, ancho, bandas)
# Es la forma que espera matplotlib.imshow para mostrar una RGB
rgb = np.dstack([banda1, banda2, banda3])
print('shape:', rgb.shape, '← matplotlib quiere así')


> **A recordar:**
> - **Rasterio** entrega y espera `(bandas, alto, ancho)` → usa `np.stack`.
> - **Matplotlib `imshow`** quiere `(alto, ancho, bandas)` para RGB → usa `np.dstack`.
> - Para pasar entre uno y otro: `np.moveaxis(arr, 0, -1)` o `arr.transpose(1, 2, 0)`.


---
## 7. Funciones universales (ufuncs)

Operan elemento a elemento sobre arrays. Las que más usaremos:


In [ ]:
x = np.array([0.01, 0.1, 1.0, 10.0, 100.0])

print('np.log(x)   =', np.log(x).round(2))    # útil en transformaciones radiométricas
print('np.sqrt(x)  =', np.sqrt(x).round(2))
print('np.abs(-x)  =', np.abs(-x))
print('np.exp([0,1]) =', np.exp([0, 1]).round(3))


In [ ]:
# np.clip — saturar valores en un rango (útil para reflectancias fuera de [0,1] por ruido)
refl = np.array([-0.05, 0.2, 0.7, 1.15, 0.95])
refl_clip = np.clip(refl, 0, 1)
print('Original:', refl)
print('Saturada:', refl_clip)


---
## 8. Estadística por eje

Cuando trabajamos con un *stack* multibanda, `axis` decide si calculamos sobre bandas, filas o columnas.


In [ ]:
# Volvemos a nuestro NDVI 4x4
print('NDVI:')
print(ndvi.round(2))

print('\nGlobal:')
print(f'  min:    {ndvi.min():.2f}')
print(f'  max:    {ndvi.max():.2f}')
print(f'  media:  {ndvi.mean():.2f}')
print(f'  mediana:{np.median(ndvi):.2f}')
print(f'  std:    {ndvi.std():.2f}')


In [ ]:
# Por filas vs por columnas
print('Media por columna (axis=0):', ndvi.mean(axis=0).round(2))
print('Media por fila    (axis=1):', ndvi.mean(axis=1).round(2))


In [ ]:
# Stack de 3 bandas → media a lo largo del eje de bandas (axis=0)
# Es lo que hacen los compuestos temporales: la media de varias fechas píxel a píxel
stack_temporal = np.stack([
    np.array([[0.3, 0.4], [0.5, 0.6]]),    # primavera
    np.array([[0.7, 0.8], [0.6, 0.5]]),    # verano
    np.array([[0.5, 0.5], [0.4, 0.3]]),    # otoño
])
print('shape:', stack_temporal.shape, '(fechas, alto, ancho)')
print('\nMedia anual por píxel:\n', stack_temporal.mean(axis=0).round(2))
print('\nMáximo anual por píxel (NDVImax):\n', stack_temporal.max(axis=0).round(2))


> El **NDVI máximo anual** (max compositing) es uno de los compuestos más utilizados — es exactamente `stack.max(axis=0)`.

---

### ⚠️ ¿Qué pasa si cambias `axis=0` por `axis=1` o `axis=2`?

Es la trampa clásica con `axis`. El array `stack_temporal` tiene forma `(3, 2, 2)` y los ejes son **`(fechas, filas, columnas)`**. `axis=N` significa "**colapsa** ese eje".

| `axis` | Colapsa | Forma resultado | Significado |
|--------|---------|-----------------|--------------|
| `0` | fechas    | `(2, 2)` | **Compuesto temporal**: media/máximo por píxel a lo largo del año |
| `1` | filas     | `(3, 2)` | Para cada fecha, media de cada columna (perfil N–S) |
| `2` | columnas  | `(3, 2)` | Para cada fecha, media de cada fila (perfil E–O) |

Con `axis=1`, el resultado concreto sería:

```python
stack_temporal.mean(axis=1)
# →
# array([[0.40, 0.50],     # primavera: media de [0.3,0.5] y de [0.4,0.6]
#        [0.65, 0.65],     # verano:    media de [0.7,0.6] y de [0.8,0.5]
#        [0.45, 0.40]])    # otoño:     media de [0.5,0.4] y de [0.5,0.3]
```

Sigue siendo una operación válida, pero **deja de ser un compuesto temporal**: pasas a "media de columnas para cada fecha", que rara vez tiene sentido en teledetección salvo para extraer perfiles transversales.

**Regla mnemotécnica:** *"el eje que pongas en `axis=` es el que **desaparece** de la forma del resultado".*


---
## 9. Máscaras booleanas — el corazón de la teledetección

Una **máscara** es un array de `True`/`False` del mismo tamaño que el raster. Te permite:
- **filtrar** píxeles (`raster[mascara]`)
- **reasignar** valores (`raster[mascara] = nuevo_valor`)
- **contar** (`mascara.sum()` cuenta los `True`)
- **combinar** (`mask1 & mask2`, `mask1 | mask2`, `~mask1`)


In [ ]:
# Máscara de agua a partir de NDVI: agua = NDVI < 0
agua = ndvi < 0
print('Máscara de agua:')
print(agua)
print(f'\nNº de píxeles de agua: {agua.sum()} de {agua.size}')


In [ ]:
# Filtrar: devuelve un VECTOR con los valores donde la máscara es True
valores_agua = ndvi[agua]
print('Valores NDVI donde hay agua:', valores_agua)
print('Media:', valores_agua.mean().round(3))


In [ ]:
# Reasignar: poner a -9999 todos los píxeles que no son vegetación
ndvi_solo_veg = ndvi.copy()                  # ¡copy! si no, modificarías el original
ndvi_solo_veg[ndvi_solo_veg < 0.3] = -9999
print(ndvi_solo_veg.round(2))


In [ ]:
# Combinar máscaras: vegetación moderada = 0.3 < NDVI <= 0.6
veg_moderada = (ndvi > 0.3) & (ndvi <= 0.6)
print(veg_moderada)
print(f'\nPíxeles de vegetación moderada: {veg_moderada.sum()}')

# Operadores: & (and), | (or), ~ (not). OJO con los paréntesis.


In [ ]:
# np.where(condición, valor_si_True, valor_si_False) — el "IF" vectorizado
clasif = np.where(ndvi < 0, 0,                       # agua
           np.where(ndvi < 0.3, 1,                   # suelo
            np.where(ndvi < 0.6, 2, 3)))             # veg moderada / sana
print(clasif)


> Este `np.where` anidado es la versión "casera" de una clasificación raster por umbrales. Es exactamente lo que harías en una calculadora de raster de QGIS, pero en una sola línea.


---
## 10. Mini-ejercicios

> **Cómo trabajar los ejercicios:** edita la celda marcada con `# *** TU CÓDIGO AQUÍ ***`. La celda siguiente contiene la solución comentada.


### Ejercicio 1 — NDVI a mano

Calcula el NDVI a partir de las bandas `rojo_e` y `nir_e` que ya tienes definidas. Pista: usa álgebra de bandas y conviértelas a `float32` si no lo están.


In [ ]:
# Datos del ejercicio
rojo_e = np.array([
    [0.05, 0.08, 0.04, 0.06],
    [0.20, 0.22, 0.25, 0.18],
    [0.30, 0.32, 0.35, 0.28],
], dtype='float32')

nir_e = np.array([
    [0.04, 0.06, 0.05, 0.04],   # agua
    [0.55, 0.60, 0.62, 0.50],   # vegetación
    [0.32, 0.34, 0.36, 0.30],   # suelo
], dtype='float32')

# *** TU CÓDIGO AQUÍ ***
ndvi_e = None

print(ndvi_e)


In [ ]:
# SOLUCIÓN — descomenta para ver
# ndvi_e = (nir_e - rojo_e) / (nir_e + rojo_e)
# print(ndvi_e.round(2))


### Ejercicio 2 — Máscara de agua

Crea una máscara booleana `mask_agua` que sea `True` donde el NDVI calculado en el ejercicio anterior es negativo. Cuenta los píxeles de agua.


In [ ]:
# *** TU CÓDIGO AQUÍ ***
mask_agua = None

print(mask_agua)
# print(f'Píxeles de agua: {mask_agua.sum()}')


In [ ]:
# SOLUCIÓN
# mask_agua = ndvi_e < 0
# print(mask_agua)
# print(f'Píxeles de agua: {mask_agua.sum()}')


### Ejercicio 3 — Índice de clorofila CIgreen (proxy)

El índice **Chlorophyll Index green** se calcula como:

$$ CI_{green} = \frac{NIR}{Green} - 1 $$

Calcúlalo con los siguientes datos. Devuelve una matriz `cigreen`.


In [ ]:
green_e = np.array([
    [0.06, 0.07, 0.06, 0.06],
    [0.08, 0.09, 0.08, 0.08],
    [0.12, 0.13, 0.14, 0.11],
], dtype='float32')

# nir_e ya está definido arriba

# *** TU CÓDIGO AQUÍ ***
cigreen = None

print(cigreen)


In [ ]:
# SOLUCIÓN
# cigreen = nir_e / green_e - 1
# print(cigreen.round(2))


### Ejercicio 4 — Estadística zonal "casera"

Tienes un raster de NDVI `ndvi_grande` (10×10) y una máscara `zona` que marca con `True` los píxeles dentro de un término municipal ficticio. Calcula la media, mínimo y máximo del NDVI **dentro de la zona**.


In [ ]:
np.random.seed(7)
ndvi_grande = (np.random.rand(10, 10) * 1.4 - 0.2).astype('float32')   # rango aprox [-0.2, 1.2]

# zona = un cuadrado en la esquina superior izquierda
zona = np.zeros_like(ndvi_grande, dtype=bool)
zona[1:5, 1:6] = True

print('NDVI:')
print(ndvi_grande.round(2))
print('\nZona (True = dentro):')
print(zona.astype(int))


In [ ]:
# *** TU CÓDIGO AQUÍ ***
# Calcula media, min y max del NDVI dentro de la zona
media_zona = None
min_zona   = None
max_zona   = None

# print(f'media={media_zona:.2f} | min={min_zona:.2f} | max={max_zona:.2f}')


In [ ]:
# SOLUCIÓN
# valores = ndvi_grande[zona]
# media_zona = valores.mean()
# min_zona   = valores.min()
# max_zona   = valores.max()
# print(f'media={media_zona:.2f} | min={min_zona:.2f} | max={max_zona:.2f}')


---
## Resumen

| Concepto | Lo clave |
|----------|----------|
| **Crear** | `np.array`, `np.zeros`, `np.ones`, `np.full`, `np.random.rand` |
| **Atributos** | `.shape`, `.ndim`, `.dtype`, `.size`, `.nbytes` |
| **Tipo** | `.astype('float32')` antes de dividir; `uint8` para máscaras |
| **Slicing** | `a[fila, col]`, `a[banda, fila, col]`, `a[i:j, k:l]` |
| **Álgebra** | `(nir - red) / (nir + red)` → vectorizado, sin bucles |
| **Apilar** | `np.stack` (bandas-alto-ancho) · `np.dstack` (alto-ancho-bandas) |
| **Stats por eje** | `arr.mean(axis=0)`, `arr.max(axis=0)` → compuestos temporales |
| **Máscaras** | `mask = arr > umbral`; `arr[mask]`; `mask1 & mask2`; `~mask` |
| **`np.where`** | El "IF" vectorizado para clasificar |

**Siguiente notebook:** abrimos una imagen Landsat real sobre Doñana con `rasterio` y aplicamos todo esto.

---
*CHG — Curso de Python para el Análisis Espacial*
